# 04. CatBoost Baseline

This notebook establishes a reproducible CatBoost baseline on the minimally prepared dataset, evaluates threshold-dependent and probability-based metrics, and persists experiment artifacts.

In [21]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

## Environment Setup

In [22]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project root configured.")

PROCESSED_DIR = PROJECT_ROOT / "data/processed"
print("Processed directory configured.")

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
EXPERIMENTS_DIR = ARTIFACTS_DIR / "experiments"

EXPERIMENT_ID = "catboost_v3_targeted_missingness_cv5"
RUN_TRAINING = True

Project root configured.
Processed directory configured.


In [23]:
from src.churn_ml.features import load_dataset
from src.churn_ml.modeling import (
    ExperimentConfig,
    load_experiment,
    save_experiment_result,
    summarize_target,
    build_experiment_summary,
)
from src.churn_ml.models.catboost import (
    CatBoostConfig,
    run_catboost_experiment,
)
from src.churn_ml.metrics import (
    evaluate_cross_fitted_thresholds,
    optimize_threshold_by_folds,
    summarize_threshold_plateau,
)

## Dataset Loading

In [24]:
dataset = load_dataset(
    version="v3_targeted_missingness",
    processed_dir=PROCESSED_DIR,
)

print(f"Dataset version: {dataset.version}")
print(f"Train shape:     {dataset.X_train.shape}")
print(f"Target shape:    {dataset.y_train.shape}")
print(f"Test shape:      {dataset.X_test.shape}")

Dataset version: v3_targeted_missingness
Train shape:     (10000, 217)
Target shape:    (10000,)
Test shape:      (2500, 217)


## Target Distribution

In [25]:
target_summary = summarize_target(dataset.y_train)

display(target_summary.style.format({"proportion": "{:.2%}"}))

,class,count,proportion
0,0,8695,86.95%
1,1,1305,13.05%


## CatBoost Baseline

In [26]:
catboost_config = CatBoostConfig.default()
experiment_config = ExperimentConfig.default()

if RUN_TRAINING:
    catboost_experiment = run_catboost_experiment(
        dataset=dataset,
        config=experiment_config,
        model_config=catboost_config,
        experiment_id=EXPERIMENT_ID,
    )

    save_experiment_result(
        result=catboost_experiment.result,
        experiments_dir=EXPERIMENTS_DIR,
        fold_metrics=catboost_experiment.fold_metrics,
        oof_predictions=catboost_experiment.oof_predictions,
        test_predictions=catboost_experiment.test_predictions,
        model=catboost_experiment.fitted_models,
        overwrite=True,
    )
else:
    catboost_experiment = load_experiment(
        experiment_id=EXPERIMENT_ID,
        experiments_dir=EXPERIMENTS_DIR,
    )

Starting CatBoost cross-validation: 5 folds, 10,000 rows, 217 features


CatBoost CV: 100%|██████████| 5/5 [34:26<00:00, 413.38s/fold, balanced_accuracy=0.7816, best_iteration=928, roc_auc=0.9582]


## Experiment Results

In [27]:
display(catboost_experiment.fold_metrics)

display(
    pd.Series(
        catboost_experiment.result.metrics,
        name="value",
    ).to_frame()
)

,fold,balanced_accuracy,roc_auc,average_precision,log_loss,decision_threshold,training_time_seconds,best_iteration
0,1,0.791950,0.969723,0.851847,0.150260,0.5,70.877949,681
1,2,0.781412,0.957390,0.814101,0.165730,0.5,98.015975,975
2,3,0.810338,0.962809,0.828956,0.157510,0.5,126.868388,904
3,4,0.826526,0.965619,0.842771,0.151306,0.5,489.055781,999
4,5,0.781603,0.958233,0.807144,0.166489,0.5,1281.096222,928


,value
balanced_accuracy,0.798366
roc_auc,0.962510
average_precision,0.827055
log_loss,0.158259
decision_threshold,0.500000
optimized_threshold_oof,0.113000
optimized_balanced_accuracy_oof,0.894018
balanced_accuracy_mean,0.798366
balanced_accuracy_std,0.019659
roc_auc_mean,0.962755


In [28]:
summary = build_experiment_summary(catboost_experiment.result)

display(summary.style.format({"value": "{:.4f}"}))

,metric,value
0,Balanced Accuracy @ default threshold,0.7984
1,Optimized OOF Balanced Accuracy,0.8940
2,Optimized OOF threshold,0.1130
3,ROC-AUC,0.9625
4,Average Precision,0.8271
5,Log Loss,0.1583
6,"Training time, minutes",34.4484


In [29]:
fold_columns = [
    "fold",
    "balanced_accuracy",
    "roc_auc",
    "average_precision",
    "log_loss",
    "best_iteration",
    "training_time_seconds",
]

display(
    catboost_experiment.fold_metrics[fold_columns].style.format(
        {
            "balanced_accuracy": "{:.4f}",
            "roc_auc": "{:.4f}",
            "average_precision": "{:.4f}",
            "log_loss": "{:.4f}",
            "training_time_seconds": "{:.1f}",
        }
    )
)

,fold,balanced_accuracy,roc_auc,average_precision,log_loss,best_iteration,training_time_seconds
0,1,0.7920,0.9697,0.8518,0.1503,681,70.9
1,2,0.7814,0.9574,0.8141,0.1657,975,98.0
2,3,0.8103,0.9628,0.8290,0.1575,904,126.9
3,4,0.8265,0.9656,0.8428,0.1513,999,489.1
4,5,0.7816,0.9582,0.8071,0.1665,928,1281.1


## Threshold Stability

In [30]:
fold_thresholds = optimize_threshold_by_folds(
    fold_metrics=catboost_experiment.fold_metrics,
    oof_predictions=catboost_experiment.oof_predictions,
)

display(
    fold_thresholds.style.format(
        {
            "optimized_threshold": "{:.3f}",
            "optimized_balanced_accuracy": "{:.4f}",
        }
    )
)

print(f"Threshold mean: {fold_thresholds['optimized_threshold'].mean():.3f}")

print(f"Threshold std:  {fold_thresholds['optimized_threshold'].std(ddof=1):.3f}")

,fold,optimized_threshold,optimized_balanced_accuracy
0,1,0.151,0.9115
1,2,0.094,0.8829
2,3,0.148,0.9029
3,4,0.183,0.9012
4,5,0.109,0.8870


Threshold mean: 0.137
Threshold std:  0.036


## Cross-Fitted Threshold Evaluation

For each holdout fold, the decision threshold is optimized using the remaining folds and then evaluated on the unseen holdout fold. This avoids evaluating a threshold on the same labels used to select it.

In [31]:
cross_fitted_result = evaluate_cross_fitted_thresholds(
    catboost_experiment.oof_predictions
)

cross_fitted_fold_metrics = cross_fitted_result.fold_metrics

display(
    cross_fitted_fold_metrics.style.format(
        {
            "threshold_cross_fitted": "{:.3f}",
            "balanced_accuracy_cross_fitted": "{:.4f}",
            "threshold_fold_optimal": "{:.3f}",
            "balanced_accuracy_fold_optimal": "{:.4f}",
            "balanced_accuracy_regret": "{:.4f}",
        }
    )
)

,fold,threshold_cross_fitted,balanced_accuracy_cross_fitted,threshold_fold_optimal,balanced_accuracy_fold_optimal,balanced_accuracy_regret
0,1,0.113,0.9056,0.151,0.9115,0.0059
1,2,0.119,0.8756,0.094,0.8829,0.0073
2,3,0.158,0.8976,0.148,0.9029,0.0054
3,4,0.113,0.8981,0.183,0.9012,0.0031
4,5,0.145,0.8794,0.109,0.8870,0.0077


In [32]:
optimized_oof_balanced_accuracy = catboost_experiment.result.metrics[
    "optimized_balanced_accuracy_oof"
]

optimism_gap = optimized_oof_balanced_accuracy - cross_fitted_result.balanced_accuracy

print(
    f"Cross-fitted OOF Balanced Accuracy: {cross_fitted_result.balanced_accuracy:.4f}"
)
print(
    "Mean cross-fitted threshold:         "
    f"{cross_fitted_fold_metrics['threshold_cross_fitted'].mean():.3f}"
)
print(
    "Cross-fitted threshold std:          "
    f"{cross_fitted_fold_metrics['threshold_cross_fitted'].std(ddof=1):.3f}"
)
print(
    "Mean Balanced Accuracy regret:       "
    f"{cross_fitted_fold_metrics['balanced_accuracy_regret'].mean():.4f}"
)
print(f"OOF threshold-selection optimism:    {optimism_gap:.4f}")

Cross-fitted OOF Balanced Accuracy: 0.8913
Mean cross-fitted threshold:         0.130
Cross-fitted threshold std:          0.021
Mean Balanced Accuracy regret:       0.0058
OOF threshold-selection optimism:    0.0028


## Threshold Plateau

This section evaluates how wide the near-optimal threshold region is. A wide plateau indicates that model performance is not sensitive to small threshold changes.

In [33]:
threshold_plateau = summarize_threshold_plateau(
    y_true=catboost_experiment.oof_predictions["target"],
    probabilities=catboost_experiment.oof_predictions["probability"].to_numpy(),
    tolerance=0.001,
)

threshold_plateau_summary = pd.DataFrame(
    [
        {
            "best_threshold": threshold_plateau.best_threshold,
            "best_balanced_accuracy": (threshold_plateau.best_balanced_accuracy),
            "lower_threshold": threshold_plateau.lower_threshold,
            "upper_threshold": threshold_plateau.upper_threshold,
            "midpoint_threshold": (threshold_plateau.midpoint_threshold),
            "plateau_width": threshold_plateau.width,
            "tolerance": threshold_plateau.tolerance,
        }
    ]
)

display(
    threshold_plateau_summary.style.format(
        {
            "best_threshold": "{:.3f}",
            "best_balanced_accuracy": "{:.4f}",
            "lower_threshold": "{:.3f}",
            "upper_threshold": "{:.3f}",
            "midpoint_threshold": "{:.3f}",
            "plateau_width": "{:.3f}",
            "tolerance": "{:.4f}",
        }
    )
)

,best_threshold,best_balanced_accuracy,lower_threshold,upper_threshold,midpoint_threshold,plateau_width,tolerance
0,0.113,0.8940,0.108,0.164,0.136,0.056,0.0010


In [34]:
fold_plateau_records = []

for fold_number in sorted(catboost_experiment.oof_predictions["fold"].unique()):
    fold_data = catboost_experiment.oof_predictions.loc[
        catboost_experiment.oof_predictions["fold"] == fold_number
    ]

    fold_plateau = summarize_threshold_plateau(
        y_true=fold_data["target"],
        probabilities=fold_data["probability"].to_numpy(),
        tolerance=0.001,
    )

    fold_plateau_records.append(
        {
            "fold": int(fold_number),
            "best_threshold": fold_plateau.best_threshold,
            "lower_threshold": fold_plateau.lower_threshold,
            "upper_threshold": fold_plateau.upper_threshold,
            "midpoint_threshold": (fold_plateau.midpoint_threshold),
            "plateau_width": fold_plateau.width,
            "best_balanced_accuracy": (fold_plateau.best_balanced_accuracy),
        }
    )

fold_plateaus = pd.DataFrame(fold_plateau_records)

display(
    fold_plateaus.style.format(
        {
            "best_threshold": "{:.3f}",
            "lower_threshold": "{:.3f}",
            "upper_threshold": "{:.3f}",
            "midpoint_threshold": "{:.3f}",
            "plateau_width": "{:.3f}",
            "best_balanced_accuracy": "{:.4f}",
        }
    )
)

,fold,best_threshold,lower_threshold,upper_threshold,midpoint_threshold,plateau_width,best_balanced_accuracy
0,1,0.151,0.123,0.153,0.138,0.030,0.9115
1,2,0.094,0.090,0.099,0.095,0.009,0.8829
2,3,0.148,0.146,0.148,0.147,0.002,0.9029
3,4,0.183,0.120,0.183,0.152,0.063,0.9012
4,5,0.109,0.109,0.113,0.111,0.004,0.8870


In [35]:
selected_threshold = catboost_experiment.result.metrics["optimized_threshold_oof"]

selected_threshold_inside_plateau = (
    threshold_plateau.lower_threshold
    <= selected_threshold
    <= threshold_plateau.upper_threshold
)

print(f"Selected OOF threshold:       {selected_threshold:.3f}")
print(
    "Near-optimal plateau:        "
    f"[{threshold_plateau.lower_threshold:.3f}, "
    f"{threshold_plateau.upper_threshold:.3f}]"
)
print(f"Selected threshold in plateau: {selected_threshold_inside_plateau}")

Selected OOF threshold:       0.113
Near-optimal plateau:        [0.108, 0.164]
Selected threshold in plateau: True


## Baseline Conclusion

The CatBoost baseline produces strong probability estimates, but the default classification threshold of 0.5 is inappropriate for the imbalanced target.

Cross-fitted threshold evaluation confirms that the performance gain from threshold optimization is stable and is not primarily caused by threshold overfitting. The selected threshold of 0.174 lies inside the global near-optimal plateau and is retained for test predictions.

In [36]:
final_threshold_summary = pd.DataFrame(
    [
        {
            "metric": "Balanced Accuracy @ 0.5",
            "value": catboost_experiment.result.metrics["balanced_accuracy"],
        },
        {
            "metric": "Optimized OOF Balanced Accuracy",
            "value": catboost_experiment.result.metrics[
                "optimized_balanced_accuracy_oof"
            ],
        },
        {
            "metric": "Cross-fitted Balanced Accuracy",
            "value": cross_fitted_result.balanced_accuracy,
        },
        {
            "metric": "Selected threshold",
            "value": selected_threshold,
        },
        {
            "metric": "Cross-fitted threshold mean",
            "value": cross_fitted_fold_metrics["threshold_cross_fitted"].mean(),
        },
        {
            "metric": "Threshold plateau lower bound",
            "value": threshold_plateau.lower_threshold,
        },
        {
            "metric": "Threshold plateau upper bound",
            "value": threshold_plateau.upper_threshold,
        },
        {
            "metric": "ROC-AUC",
            "value": catboost_experiment.result.metrics["roc_auc"],
        },
        {
            "metric": "Average Precision",
            "value": catboost_experiment.result.metrics["average_precision"],
        },
    ]
)

display(final_threshold_summary.style.format({"value": "{:.4f}"}))

,metric,value
0,Balanced Accuracy @ 0.5,0.7984
1,Optimized OOF Balanced Accuracy,0.8940
2,Cross-fitted Balanced Accuracy,0.8913
3,Selected threshold,0.1130
4,Cross-fitted threshold mean,0.1296
5,Threshold plateau lower bound,0.1080
6,Threshold plateau upper bound,0.1640
7,ROC-AUC,0.9625
8,Average Precision,0.8271


In [37]:
expected_test_predictions = (
    catboost_experiment.test_predictions["probability"] >= selected_threshold
).astype(int)

assert expected_test_predictions.equals(
    catboost_experiment.test_predictions["prediction_optimized_oof"]
)

test_positive_rate = expected_test_predictions.mean()

print(f"Selected threshold:       {selected_threshold:.3f}")
print(f"Test positive rate:       {test_positive_rate:.2%}")
print(f"Predicted positive rows:  {expected_test_predictions.sum():,}")
print(f"Total test rows:          {len(expected_test_predictions):,}")

Selected threshold:       0.113
Test positive rate:       24.20%
Predicted positive rows:  605
Total test rows:          2,500


In [38]:
experiment_dir = EXPERIMENTS_DIR / EXPERIMENT_ID
experiment_dir.mkdir(parents=True, exist_ok=True)

cross_fitted_fold_metrics.to_csv(
    experiment_dir / "cross_fitted_threshold_metrics.csv",
    index=False,
)

fold_plateaus.to_csv(
    experiment_dir / "threshold_plateaus_by_fold.csv",
    index=False,
)

final_threshold_summary.to_csv(
    experiment_dir / "threshold_summary.csv",
    index=False,
)

print(f"Threshold diagnostics saved to: {experiment_dir}")

Threshold diagnostics saved to: c:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\artifacts\experiments\catboost_v3_targeted_missingness_cv5


### Interpretation

- The default threshold of 0.5 produces a Balanced Accuracy of approximately 0.794.
- Global OOF threshold optimization increases Balanced Accuracy to approximately 0.895.
- Cross-fitted evaluation produces approximately 0.893, indicating only a small threshold-selection optimism of about 0.002.
- Cross-fitted thresholds are highly stable and concentrate around 0.174–0.176.
- The selected threshold of 0.174 lies inside the global near-optimal interval of 0.168–0.176.
- The CatBoost baseline is therefore retained with a decision threshold of 0.174.

## Feature Importance

CatBoost feature importance is calculated using `PredictionValuesChange`
for each fitted fold and then averaged across folds.

The values are normalized within each fold. The resulting rankings are
used to assess feature relevance and stability across validation folds.

In [39]:
feature_importance_by_fold = []

for fold_number, model in enumerate(
    catboost_experiment.fitted_models,
    start=1,
):
    raw_importance = np.asarray(
        model.get_feature_importance(type="PredictionValuesChange"),
        dtype=float,
    )

    if len(raw_importance) != dataset.X_train.shape[1]:
        raise ValueError(
            "Feature importance length does not match the number of input features."
        )

    fold_importance = pd.DataFrame(
        {
            "feature": dataset.X_train.columns,
            "importance": raw_importance,
            "fold": fold_number,
        }
    )

    total_importance = fold_importance["importance"].sum()

    if total_importance > 0:
        fold_importance["importance"] /= total_importance

    feature_importance_by_fold.append(fold_importance)

feature_importance_folds = pd.concat(
    feature_importance_by_fold,
    ignore_index=True,
)

feature_importance_summary = (
    feature_importance_folds.groupby(
        "feature",
        as_index=False,
    )
    .agg(
        importance_mean=("importance", "mean"),
        importance_std=("importance", "std"),
        folds_used=(
            "importance",
            lambda values: int((values > 0).sum()),
        ),
    )
    .sort_values(
        "importance_mean",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    feature_importance_summary.head(30).style.format(
        {
            "importance_mean": "{:.4%}",
            "importance_std": "{:.4%}",
        }
    )
)

,feature,importance_mean,importance_std,folds_used
0,Var126,27.6733%,2.2498%,5
1,Var212,11.1170%,0.7440%,5
2,Var126_is_missing,9.1491%,2.4081%,5
3,Var73,7.9508%,1.8395%,5
4,Var74,3.7793%,0.5264%,5
5,Var228,3.4757%,1.1579%,5
6,Var193,2.1271%,0.5014%,5
7,Var144,2.1206%,0.2342%,5
8,Var199,2.1121%,0.2841%,5
9,Var57,1.9352%,0.2115%,5


In [40]:
feature_importance_path = experiment_dir / "feature_importance.csv"

feature_importance_summary.to_csv(
    feature_importance_path,
    index=False,
)

print(f"Feature importance saved to: {feature_importance_path}")

Feature importance saved to: c:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\artifacts\experiments\catboost_v3_targeted_missingness_cv5\feature_importance.csv
